In [1]:
import torch
from transformers import pipeline
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

if torch.backends.mps.is_available():
    device = "mps"
    pipeline_device = torch.device("mps")
else:
    device = "cpu"
    pipeline_device = -1

print(f"Using device: {device}")

Using device: mps


In [2]:
model_name = "textattack/roberta-base-MRPC"

clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True,
    max_length=128,
    return_all_scores=False
)

print(f"Loaded pipeline model: {model_name}")

config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-MRPC
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pipeline model: textattack/roberta-base-MRPC


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [ ]:
pair_inputs = [{"text": row["sentence1"], "text_pair": row["sentence2"]} for row in dataset]
labels = dataset["label"]

print("Prepared true sentence-pair inputs for pipeline inference.")
print(pair_inputs[0])

In [ ]:
batch_size = 32
raw_outputs = clf(pair_inputs, batch_size=batch_size)

label_to_id = {k.upper(): v for k, v in clf.model.config.label2id.items()}
predictions = [label_to_id[o["label"].upper()] for o in raw_outputs]
confidences = [float(o["score"]) for o in raw_outputs]

print(f"Completed pipeline inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
cm = confusion_matrix(labels, predictions)
avg_confidence = sum(confidences) / len(confidences)

print("Evaluation metrics:")
print(f"Accuracy              : {accuracy:.4f}")
print(f"Precision             : {precision:.4f}")
print(f"Recall                : {recall:.4f}")
print(f"F1                    : {f1:.4f}")
print(f"Average confidence    : {avg_confidence:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
errors = []

for i, (true_label, pred_label, conf) in enumerate(zip(labels, predictions, confidences)):
    if true_label != pred_label:
        row = dataset[i]
        errors.append({
            "index": i,
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"],
            "true_label": true_label,
            "pred_label": pred_label,
            "confidence": conf,
        })

errors = sorted(errors, key=lambda x: x["confidence"], reverse=True)
num_examples_to_show = min(5, len(errors))

print(f"Total errors: {len(errors)}")
print(f"Showing {num_examples_to_show} highest-confidence errors")

for example in errors[:num_examples_to_show]:
    print(f"Index: {example['index']}")
    print(f"sentence1: {example['sentence1']}")
    print(f"sentence2: {example['sentence2']}")
    print(f"true label: {example['true_label']} ({label_map[example['true_label']]})")
    print(f"pred label: {example['pred_label']} ({label_map[example['pred_label']]})")
    print(f"confidence: {example['confidence']:.4f}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("inference_method=transformers.pipeline_text_classification")
print("input_format=true_sentence_pair_dict_with_text_and_text_pair")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"average_confidence={avg_confidence:.4f}")
print(f"num_errors={len(errors)}")